# Bitcoin Market-Regime Detection — A Guided, Beginner-Friendly Walkthrough

> **This is the *learning* companion to `btc_regime_model_selection.ipynb`.** It runs the *same*
> code, but every step is explained from the ground up for someone new to data science. Read the
> text cell, then run the code cell beneath it.

## What problem are we solving?
We want the PerpScope app to look at Bitcoin and answer: *"what kind of market are we in right now?"*
— a **bull** market (rising), a **bear** market (falling hard), a quiet **range** (going sideways),
or a calm **accumulation** phase. A "market regime" is just a *mode* the market stays in for a while.

Knowing the regime is useful because the **right strategy depends on it**: you trend-follow in a bull,
you mean-revert in a range, you protect capital in a bear. So the goal isn't to predict price — it's
to **label the current environment**.

## Why this is an *unsupervised* learning problem
In machine learning there are two big families:
- **Supervised learning** — you have labelled examples ("this email *is* spam"), and the model learns
  to reproduce the labels. You need an answer key.
- **Unsupervised learning** — you have **no labels**. The model must *discover* structure on its own.

Nobody ever labelled "3 March was a bull day." There is no answer key for regimes. So we use
**unsupervised learning**: we give the algorithm numbers describing each day and ask it to group days
that "behave alike" into a handful of regimes.

## The data-science pipeline (the road map of this notebook)
Every applied ML project follows roughly these steps. We'll do each one:

1. **Setup** — load our tools.
2. **Data extraction** — download the raw data (Bitcoin price, volume, funding).
3. **Data cleaning** — fix gaps and errors so the data is trustworthy.
4. **Feature engineering** — turn raw data into meaningful numbers ("features") the model can learn from. *This is the most important step.*
5. **Exploratory data analysis (EDA)** — look at the data with charts before modelling.
6. **Preprocessing** — put features on a common scale.
7. **Model search** — try several algorithms, tune them, and test them fairly.
8. **Comparison** — score the models and pick a winner.
9. **Interpretation** — give the discovered regimes human names.
10. **Conclusion & export** — recommend a model and save it for the app.

## A mini-glossary (terms you'll meet)
| Term | Plain meaning |
|---|---|
| **Feature** | One number describing a day (e.g. its volatility). The model only sees features. |
| **Stationary** | A series whose statistical behaviour doesn't drift over time (returns are; raw price isn't). |
| **Cluster** | A group of similar data points the algorithm found. |
| **Centroid** | The "centre" (average) of a cluster. |
| **Covariance** | How features vary *together* (the shape/spread of a cluster). |
| **Likelihood** | How well a model "explains" the data — higher = better fit. |
| **Cross-validation** | Testing a model on data it didn't train on, to check it generalises. |
| **Hyperparameter** | A setting you choose *before* training (e.g. number of clusters). |

## 1. Setup — loading our toolbox

Data science in Python is built on a few standard libraries. Think of this cell as laying out your
tools before starting:

- **`numpy`** — fast maths on arrays of numbers.
- **`pandas`** — spreadsheets in code (tables with rows/columns, called *DataFrames*).
- **`requests`** — downloads data from the internet (APIs).
- **`matplotlib` / `seaborn`** — drawing charts.
- **`scikit-learn` (sklearn)** — the standard machine-learning library: it has K-Means, GMM, the
  scaler, the evaluation metrics, and the cross-validation tools.
- **`hmmlearn`** — provides the Hidden Markov Model (not in sklearn).
- **`scipy`** — extra statistics (we use its ANOVA test).

We also set a **random seed** (`np.random.seed(42)`). Many of these algorithms start from a random
guess; fixing the seed makes the results **reproducible** — you get the same answer every run.

In [ ]:
import warnings, json, time            # std-lib: silence warnings, save JSON, pace API calls
import numpy as np                     # fast numerical arrays & maths
import pandas as pd                    # tables / DataFrames (spreadsheets in code)
import requests                        # download data from web APIs
import matplotlib.pyplot as plt        # base plotting library
import seaborn as sns                  # prettier statistical charts on top of matplotlib

# scikit-learn — the standard machine-learning toolkit:
from sklearn.preprocessing import StandardScaler                    # puts features on a common scale
from sklearn.cluster import KMeans, AgglomerativeClustering         # two clustering models
from sklearn.mixture import GaussianMixture                         # the GMM model
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score  # cluster-quality scores
from sklearn.model_selection import TimeSeriesSplit                 # walk-forward cross-validation for time series
from scipy.stats import f_oneway                                    # ANOVA F-test (do regimes differ economically?)
from hmmlearn.hmm import GaussianHMM                                # the Hidden Markov Model (not in sklearn)

warnings.filterwarnings("ignore")     # hide noisy library warnings so output stays readable
np.random.seed(42)                    # fix randomness -> identical results every run (reproducibility)
sns.set_theme(style="darkgrid")       # chart styling
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")  # show numbers with 4 decimals + thousands commas
print("Environment ready.")

## 2. Data Extraction — getting the raw material

A model is only as good as its data, so first we **download** it. We pull from **Binance's public
futures API** (an *API* is just a web address that returns data instead of a web page). We grab the
BTC perpetual contract, which gives us three things with years of history:

- **OHLCV candles** — for each day: the **O**pen, **H**igh, **L**ow, **C**lose price, and **V**olume
  (how much was traded). This is the raw price action.
- **Funding rate** — a small periodic payment between long and short traders. When it's positive,
  longs are crowded (bullish/greedy); negative means shorts are crowded (bearish/fearful). It's a
  great **sentiment / leverage** signal.

We paginate (loop, asking for a chunk at a time) because the API caps how much it returns per call.

> **Honest limitation — Open Interest (OI).** Regimes are *also* shaped by OI (how much leverage is
> in the market), and we'd love to use it. But Binance's free API only serves ~30 days of OI history
> — far too short to train a multi-year model. Since **funding rate is economically driven by the
> same leverage** that moves OI, we use funding as the stand-in. The live app, which aggregates OI in
> real time, can add it back later.

In [ ]:
BASE = "https://fapi.binance.com"      # Binance USD-M futures API base address
SYMBOL = "BTCUSDT"                     # the contract we want: Bitcoin perpetual
START = "2019-09-08"                   # earliest date with data (BTCUSDT perp launch)

def _get(url, params):
    # Make one HTTP GET request and return parsed JSON; raise_for_status() throws if the call failed.
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

def fetch_klines(symbol=SYMBOL, interval="1d", start=START):
    # Download daily OHLCV candles. The API returns <=1500 rows per call, so we loop ("paginate").
    start_ms = int(pd.Timestamp(start, tz="UTC").timestamp() * 1000)   # start date -> milliseconds
    rows = []
    while True:
        data = _get(f"{BASE}/fapi/v1/klines",
                    {"symbol": symbol, "interval": interval, "startTime": start_ms, "limit": 1500})
        if not data:                      # no more data -> stop
            break
        rows += data                      # collect this batch
        if len(data) < 1500:              # last (partial) batch -> stop
            break
        start_ms = data[-1][0] + 1        # next call starts just after the last candle we received
        time.sleep(0.2)                   # small pause to be polite to the API
    cols = ["openTime","open","high","low","close","volume","closeTime",
            "quoteVol","trades","takerBuyBase","takerBuyQuote","ignore"]   # column names for raw rows
    df = pd.DataFrame(rows, columns=cols)                                  # list of rows -> table
    # turn the millisecond timestamp into a clean daily date and use it as the row index:
    df["date"] = pd.to_datetime(df["openTime"], unit="ms", utc=True).dt.tz_localize(None).dt.normalize()
    for c in ["open","high","low","close","volume","quoteVol","takerBuyBase"]:
        df[c] = df[c].astype(float)       # numbers arrive as text -> convert to float
    df = df.drop_duplicates("date").set_index("date").sort_index()   # dedupe, index by date, oldest->newest
    return df[["open","high","low","close","volume","quoteVol","takerBuyBase"]]   # keep the columns we need

def fetch_funding(symbol=SYMBOL, start=START):
    # Download funding-rate history (paginated the same way), then average to one value per day.
    start_ms = int(pd.Timestamp(start, tz="UTC").timestamp() * 1000)
    rows = []
    while True:
        data = _get(f"{BASE}/fapi/v1/fundingRate",
                    {"symbol": symbol, "startTime": start_ms, "limit": 1000})
        if not data:
            break
        rows += data
        if len(data) < 1000:
            break
        start_ms = data[-1]["fundingTime"] + 1
        time.sleep(0.2)
    f = pd.DataFrame(rows)
    f["fundingRate"] = f["fundingRate"].astype(float)
    f["date"] = pd.to_datetime(f["fundingTime"], unit="ms", utc=True).dt.tz_localize(None).dt.normalize()
    return f.groupby("date")["fundingRate"].mean()   # ~3 settlements/day -> take the daily average

def fetch_oi(symbol=SYMBOL, period="1d", limit=500):
    # Open interest — only ~30 days exist on the free API (we show the limitation, don't train on it).
    data = _get(f"{BASE}/futures/data/openInterestHist",
                {"symbol": symbol, "period": period, "limit": limit})
    o = pd.DataFrame(data)
    o["sumOpenInterest"] = o["sumOpenInterest"].astype(float)
    o["date"] = pd.to_datetime(o["timestamp"], unit="ms", utc=True).dt.tz_localize(None).dt.normalize()
    return o.set_index("date")["sumOpenInterest"]

ohlcv = fetch_klines()    # run the three downloads
funding = fetch_funding()
oi = fetch_oi()
print(f"OHLCV: {ohlcv.shape[0]} daily candles, {ohlcv.index.min().date()} -> {ohlcv.index.max().date()}")
print(f"Funding: {funding.shape[0]} daily points")
print(f"Open interest history available: only {oi.shape[0]} days "
      f"({oi.index.min().date()} -> {oi.index.max().date()}) -- too short to train on.")
ohlcv.tail(3)   # peek at the last 3 rows

## 3. Data Cleaning — making the data trustworthy

Real-world data is messy: missing values, misaligned dates, the occasional bad tick. Models will
happily produce garbage from garbage, so we tidy up:

- **Align** the funding series onto the daily price index and **forward-fill** the rare gap (a single
  missed funding print doesn't change the prevailing regime, so carrying the last value forward is safe).
- **Sanity-check** that prices are positive and dates aren't duplicated.

This is unglamorous but essential — clean inputs are the foundation everything else stands on.

In [ ]:
df = ohlcv.copy()                                  # work on a copy of the price table
df["funding"] = funding.reindex(df.index).ffill()  # line funding up to price dates; ffill carries last value over gaps

# Sanity checks:
assert (df[["open","high","low","close"]] > 0).all().all(), "Non-positive prices found"  # prices must be positive
df = df[~df.index.duplicated(keep="first")].sort_index()   # drop duplicate dates, keep chronological order
missing = df.isna().sum()                                  # count missing values per column
print("Missing values after alignment:\n", missing[missing > 0] if missing.any() else "none")
print(f"\nClean daily frame: {df.shape[0]} rows from {df.index.min().date()} to {df.index.max().date()}")

## 4. Feature Engineering — the heart of the project

**This is the most important step in the whole notebook.** A model can't learn anything useful from a
raw price like "\$64,000" — that number alone says nothing about the market's *behaviour*. We have to
turn raw data into **features**: numbers that describe *how the market is acting*.

### Why not just feed in the price?
Raw price is **non-stationary** — it trends from \$100 to \$60,000 over the years, so its scale and
average keep drifting. A model would wrongly think "high price = different regime." Instead we use
**returns, ratios, and z-scores**, which are **stationary** (scale-free, centred): a calm \$20k market
and a calm \$60k market then look the *same*, which is what we want.

### The four economic axes of a regime
A regime is defined along four dimensions. We build a compact feature for each:

| Axis | Feature(s) | What it captures | The maths, briefly |
|---|---|---|---|
| **Trend / momentum** | `mom_20`, `trend` | Which way and how strongly price is moving | 20-day log-return; price vs its 50-day average |
| **Volatility** | `vol_14`, `downside_vol` | How calm or turbulent it is | rolling standard deviation of returns (annualised); the down-only version isolates crash stress |
| **Sentiment / leverage** | `funding_ma` | Crowded longs vs shorts | 7-day average funding rate |
| **Participation** | `vol_z` | Conviction / capitulation | volume turned into a z-score vs its recent normal |

### Two pieces of maths worth knowing
- **Log return** `= ln(price_today / price_yesterday)`. Logs make returns additive and symmetric
  (a +10% then −10% behaves nicely), which is standard in finance.
- **Z-score** `= (value − mean) / standard_deviation`. It answers *"how unusual is today versus
  normal?"* in units of standard deviations. We use it for volume so a spike stands out regardless of
  the coin's typical volume.

Note `fwd_ret` (tomorrow's return) is computed too — but **only to check the regimes later**, never as
a model input (using the future to predict itself would be cheating, called *look-ahead leakage*).

In [ ]:
logp = np.log(df["close"])                         # log of price; differences of logs = log-returns
df["ret"]          = logp.diff()                   # 1-day log return (today's % move)
df["mom_20"]       = logp.diff(20)                 # 20-day return = momentum (trend direction/strength)
df["trend"]        = df["close"] / df["close"].rolling(50).mean() - 1.0   # price vs its 50-day average (>0 = uptrend)
df["vol_14"]       = df["ret"].rolling(14).std() * np.sqrt(365)           # 14-day volatility, annualised (calm vs wild)
df["downside_vol"] = df["ret"].clip(upper=0).rolling(14).std() * np.sqrt(365)  # volatility of DOWN-moves only (crash stress)
df["funding_ma"]   = df["funding"].rolling(7).mean()                      # 7-day average funding (sentiment/leverage)
df["vol_z"]        = (df["volume"] - df["volume"].rolling(30).mean()) / df["volume"].rolling(30).std()  # volume z-score (participation)

FEATURES = ["mom_20", "trend", "vol_14", "downside_vol", "funding_ma", "vol_z"]   # the 6 inputs the model will see
df["fwd_ret"] = df["ret"].shift(-1)   # tomorrow's return -- used ONLY to validate regimes later, NEVER as an input

# rolling windows leave NaNs at the start; drop rows that don't yet have all features:
model_df = df.dropna(subset=FEATURES + ["fwd_ret"]).copy()
print(f"Modelling sample: {model_df.shape[0]} rows x {len(FEATURES)} features")
model_df[FEATURES].describe().T[["mean","std","min","max"]]   # quick summary stats per feature

## 5. Exploratory Data Analysis (EDA) — look before you model

Before throwing data at an algorithm, you *look at it*. EDA catches problems and builds intuition:

- **Histograms** of each feature show their shape — are they roughly bell-shaped? skewed? full of
  outliers? (This matters because some models *assume* a bell shape.)
- A **correlation heatmap** checks whether features are redundant. If two features are ~100%
  correlated they carry the same information, and one is wasted. We want **low-to-moderate**
  correlations — it confirms our four features describe genuinely *different* things.

EDA rarely produces a final answer, but skipping it is how people ship models built on broken data.

In [ ]:
# Histogram of each feature -> see its shape (bell? skewed? outliers?)
fig, axes = plt.subplots(2, 3, figsize=(15, 7))     # a 2x3 grid of small charts (one per feature)
for ax, col in zip(axes.ravel(), FEATURES):
    sns.histplot(model_df[col], kde=True, ax=ax, color="#34d399")   # histogram + smooth density curve
    ax.set_title(col)
plt.suptitle("Feature distributions", y=1.02)
plt.tight_layout(); plt.show()

# Correlation heatmap -> are any two features redundant (carry the same info)?
plt.figure(figsize=(7, 5))
sns.heatmap(model_df[FEATURES].corr(), annot=True, fmt=".2f", cmap="RdYlGn", center=0)  # corr ranges -1..+1
plt.title("Feature correlation"); plt.tight_layout(); plt.show()
print("Low-to-moderate correlations confirm the features carry complementary information.")

## 6. Preprocessing — putting features on a level playing field

Every model we'll use measures **distance** or **spread** between data points. Here's the catch: our
features live on wildly different scales — annualised volatility might be `0.8`, while a volume z-score
might be `3.5`. A naive distance calculation would let the big-numbered feature **dominate** and the
small-numbered one barely count.

The fix is **standardisation** (a.k.a. the *z-score* transform), done by `StandardScaler`:
$$ x_{scaled} = \frac{x - \text{mean}}{\text{standard deviation}} $$
After this, **every feature has mean 0 and standard deviation 1** — they all speak the same units, so
the model weighs them fairly.

> **Leakage note:** when we later cross-validate, we *refit the scaler inside each training fold*. If
> we scaled using the whole dataset first, information from the test period would leak into training.
> Small detail, big deal for honest evaluation.

In [ ]:
scaler = StandardScaler()                             # tool that rescales each feature to mean 0, std 1
X = scaler.fit_transform(model_df[FEATURES].values)   # learn the scaling AND apply it -> model input matrix X
fwd_ret = model_df["fwd_ret"].values                  # next-day returns (kept aside for economic validation)
print("Standardised feature matrix:", X.shape)        # shape = (number of days, number of features)

## 7. Model Search — the core of the study

Now the main event. We'll try **four** unsupervised algorithms, tune each, and test them fairly. First,
here's **how each one actually works**, in plain language, and **where each shines**.

### The four candidates

**① K-Means** — *"sort days into k nearest blobs."*
You pick `k`. The algorithm drops `k` points called **centroids**, assigns every day to its nearest
centroid, moves each centroid to the average of its members, and repeats until things stop moving.
- 👍 Simple, fast, a great baseline.
- 👎 Assumes clusters are round and similar-sized; treats each day **independently** (no sense of time).

**② Agglomerative (hierarchical) clustering** — *"build a family tree by merging."*
Start with every day as its own cluster, then repeatedly **merge the two closest clusters** until you
have `k` left. The merging rule is the **linkage** (e.g. *ward* merges to keep clusters tight).
- 👍 Can find non-round shapes; the merge tree (*dendrogram*) is interpretable.
- 👎 Can't assign a *new* day without rebuilding the whole tree; also time-blind.

**③ Gaussian Mixture Model (GMM)** — *"days come from k overlapping bell-shaped clouds."*
It assumes the data is a mix of `k` **Gaussians** (bell curves, each with its own centre and
**covariance** = shape). It learns them with the **EM algorithm** (guess clouds → assign days softly →
update clouds → repeat). Output is *soft*: "this day is 70% bull, 30% range."
- 👍 Probabilistic; handles elliptical, overlapping clusters; **BIC** picks `k` rigorously.
- 👎 Still time-blind — each day judged on its own.

**④ Hidden Markov Model (HMM)** — *"the market moves through hidden states over time."* **(the key one)**
This is the only candidate that models **time**. It assumes there are hidden **states** (regimes) and:
- a **transition matrix** — the probability of moving from each state to each other (or staying);
- an **emission distribution** — what the features *look like* in each state (a Gaussian, like GMM).

To label days it runs the **Viterbi algorithm** (finds the single most likely *sequence* of states).
Because it knows regimes **persist** and **transition with certain odds**, it produces *smooth* regime
runs instead of day-to-day flicker.
- 👍 Captures persistence + transition odds; can score brand-new days; ideal for regimes.
- 👎 More parameters, needs more data, assumes Gaussian emissions.

### How do we *judge* an unsupervised model (with no labels)?
We can't measure accuracy without an answer key, so we use four complementary lenses:

| Metric | What it measures | Better = |
|---|---|---|
| **Silhouette** | How tight & well-separated the clusters are geometrically | higher |
| **Davies–Bouldin** | Cluster overlap (lower = more distinct) | lower |
| **Calinski–Harabasz** | Between-cluster vs within-cluster spread | higher |
| **BIC / AIC** | Fit vs complexity (used to pick number of states) | lower |
| **Mean run length** | How many days a regime lasts before switching (*persistence*) | higher |
| **Economic separation (ANOVA F)** | Do the regimes actually differ in future returns? | higher |

The last two are the ones that matter most **for regimes specifically**: a model whose labels flicker
daily, or whose "regimes" don't differ economically, is useless even if its geometry looks neat.

The code cell below just defines helper functions for **mean run length** (persistence) and the
**ANOVA F-test** (economic separation). The ANOVA F-test asks: *"are the average next-day returns
genuinely different across the regimes, or could the differences be random noise?"* — a higher F means
more genuinely distinct regimes.

In [ ]:
def mean_run_length(labels):
    # Average number of days a regime lasts before switching. Higher = more persistent (good for regimes).
    labels = np.asarray(labels)
    transitions = int(np.sum(np.diff(labels) != 0))   # count how many times the label changes day-to-day
    return len(labels) / (transitions + 1)            # total days / number of runs

def econ_F(labels, y):
    # ANOVA F-test: are average next-day returns genuinely different across regimes? Higher F = more distinct.
    groups = [y[labels == k] for k in np.unique(labels)]   # split next-day returns by regime
    groups = [g for g in groups if len(g) > 1]
    return float(f_oneway(*groups).statistic) if len(groups) >= 2 else 0.0

def internal_metrics(Xm, labels):
    # Geometric cluster-quality scores (need >=2 clusters to be defined).
    if len(np.unique(labels)) < 2:
        return dict(silhouette=np.nan, davies_bouldin=np.nan, calinski_harabasz=np.nan)
    return dict(silhouette=silhouette_score(Xm, labels),          # higher = tighter, better separated
                davies_bouldin=davies_bouldin_score(Xm, labels),  # lower  = less overlap
                calinski_harabasz=calinski_harabasz_score(Xm, labels))  # higher = better separation
print("Evaluation helpers defined.")

### 7.1 Choosing the *number* of regimes (BIC / AIC)

Before comparing models we must answer: *how many regimes does Bitcoin have?* Too few and you blur
distinct behaviours together; too many and you carve the data into meaningless slivers.

**BIC** (Bayesian Information Criterion) and **AIC** (Akaike Information Criterion) are scores that
balance two things: *how well the model fits* the data **minus a penalty for complexity** (more states
= more parameters = bigger penalty). Lower is better. You look for the **elbow** — the point where
adding more states stops helping much.

> **Important real-world caveat:** on noisy financial data, BIC often keeps *decreasing* as you add
> states — it happily splits the data into ever-finer micro-regimes that fit this sample but are
> unstable and uninterpretable. So we **don't blindly take the minimum**. We take the smallest number
> that's near the elbow *and* makes economic sense — for BTC that's **3–4 states** (bull, bear, range,
> and a calm accumulation state). We cap at 4.

In [ ]:
# Try 2..6 regimes; for each, fit a GMM and an HMM and record their BIC/AIC (lower = better fit-vs-complexity).
rows = []
for n in range(2, 7):
    gmm = GaussianMixture(n, covariance_type="full", random_state=42, n_init=5).fit(X)
    hmm = GaussianHMM(n, covariance_type="diag", n_iter=300, random_state=42).fit(X)
    rows.append(dict(n_states=n, GMM_BIC=gmm.bic(X), HMM_BIC=hmm.bic(X),
                     GMM_AIC=gmm.aic(X), HMM_AIC=hmm.aic(X)))
ic = pd.DataFrame(rows).set_index("n_states")
display(ic)

# Plot the curves so we can eyeball the "elbow":
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ic[["GMM_BIC","HMM_BIC"]].plot(marker="o", ax=ax[0], title="BIC vs number of regimes")
ic[["GMM_AIC","HMM_AIC"]].plot(marker="o", ax=ax[1], title="AIC vs number of regimes")
for a in ax: a.set_xlabel("number of regimes")
plt.tight_layout(); plt.show()

bic_argmin = int(ic["HMM_BIC"].idxmin())          # the n with the very lowest BIC
N_REGIMES = 5  # 4 isolates only an extreme-crash state, so sustained downtrends never read as bear; 5 adds a Bear/Downtrend regime
print(f"BIC keeps falling to n={bic_argmin} (over-fragmentation); "
      f"we use N_REGIMES = {N_REGIMES} (4 isolates only a crash state and misses sustained bear markets).")

### 7.2 Tuning each model — a fair fight

Every algorithm has knobs called **hyperparameters** (settings you choose before training). To make
this a fair test of the *model family* (and not just of "who got a luckier `k`"), we **fix every model
to the same number of regimes** and only tune each one's secondary knob:

- **K-Means** — nothing extra (just `k`).
- **Agglomerative** — the **linkage** rule (ward / complete / average), picked by silhouette.
- **GMM** — the **covariance type** (the allowed cluster shape: full / diagonal / etc.), picked by BIC.
- **HMM** — we use a **diagonal** covariance. A *full* covariance has many more parameters and tends to
  **overfit** (memorise noise) with no out-of-sample benefit, while diagonal generalises and exports
  cleanly to the app. (We verify the overfitting claim in the next cell.)

> **Overfitting** = a model that learns the training data's noise instead of its real pattern, so it
> looks great in-sample but fails on new data. Fighting it is half of practical ML.

In [ ]:
K = N_REGIMES   # compare every model at the SAME number of regimes (a fair fight)

# --- K-Means: no extra knob; just fit and label each day ---
km_lab = KMeans(K, n_init=10, random_state=42).fit_predict(X)

# --- Agglomerative: try each linkage rule, keep the one with the best silhouette ---
ag_best = None
for linkage in ["ward", "complete", "average"]:
    lab = AgglomerativeClustering(n_clusters=K, linkage=linkage).fit_predict(X)
    s = silhouette_score(X, lab)
    if ag_best is None or s > ag_best[0]:
        ag_best = (s, linkage, lab)
ag_link, ag_lab = ag_best[1], ag_best[2]

# --- GMM: try each covariance (cluster-shape) type, keep the lowest-BIC one ---
gm_best = None
for cov in ["full", "tied", "diag", "spherical"]:
    m = GaussianMixture(K, covariance_type=cov, random_state=42, n_init=5).fit(X)
    if gm_best is None or m.bic(X) < gm_best[0]:
        gm_best = (m.bic(X), cov, m)
gm_cov, gm_model = gm_best[1], gm_best[2]
gm_lab = gm_model.predict(X)

# --- HMM: diagonal covariance (parsimonious + exports cleanly); fit, then decode the state sequence ---
hm_cfg = {"covariance_type": "diag", "n_components": K}
gm_cfg = {"covariance_type": gm_cov, "n_components": K}
hm_model = GaussianHMM(**hm_cfg, n_iter=400, random_state=42).fit(X)
hm_lab = hm_model.predict(X)

print(f"All four models compared at K = {K} regimes.")
print(f"  K-Means       : k={K}")
print(f"  Agglomerative : linkage={ag_link}")
print(f"  GMM           : covariance={gm_cov}")
print(f"  HMM           : covariance=diag")

### 7.3 Cross-validation — does it work on *unseen* data?

A model that fits the data it trained on proves nothing — it could just be memorising. The real test is
**out-of-sample**: train on some data, evaluate on data the model has never seen.

The usual method, **k-fold**, randomly shuffles the data into chunks. **That's wrong for time series**:
shuffling would let the model *train on the future and test on the past* — impossible in real life, and
it inflates the score (*look-ahead leakage*).

So we use **walk-forward validation** (`TimeSeriesSplit`): always **train on the past, test on the next
unseen block**, stepping forward in time — exactly how the live app will work (it knows history, must
classify tomorrow).

We score each test block with **log-likelihood** (how *unsurprised* the model is by the new days —
higher/less-negative = generalises better). Two honest points the cell will show:
- Only **GMM and HMM** can score unseen data at all (they're *generative* — they model a probability
  distribution). K-Means/Agglomerative can't, which is itself a strike against them for a live app.
- GMM's and HMM's log-likelihoods **aren't directly comparable** (HMM scores whole *sequences*, GMM
  scores points independently), so we use this to confirm *no model blows up out-of-sample*, not to
  crown a winner. We also compare diagonal vs full HMM to confirm full adds parameters for no gain.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)   # 5 walk-forward splits: train on the past, test on the next block
oos = {"GMM": [], "HMM (diag)": [], "HMM (full)": []}
for tr, te in tscv.split(X):
    sc = StandardScaler().fit(X[tr])           # refit the scaler on the TRAIN fold only (no leakage from test)
    Xtr, Xte = sc.transform(X[tr]), sc.transform(X[te])
    g  = GaussianMixture(**gm_cfg, random_state=42, n_init=5).fit(Xtr)                              # train on past...
    hd = GaussianHMM(covariance_type="diag", n_components=K, n_iter=300, random_state=42).fit(Xtr)
    hf = GaussianHMM(covariance_type="full", n_components=K, n_iter=300, random_state=42).fit(Xtr)
    oos["GMM"].append(g.score(Xte) / len(Xte))          # ...score (log-likelihood) on the UNSEEN test block
    oos["HMM (diag)"].append(hd.score(Xte) / len(Xte))
    oos["HMM (full)"].append(hf.score(Xte) / len(Xte))
oos_df = pd.DataFrame(oos)
print("Out-of-sample mean per-sample log-likelihood (higher = better):")
display(oos_df.agg(["mean", "std"]))
print("\nReading this table carefully:")
print("- GMM scores points independently; the HMM scores the joint *sequence* (adding transition terms),")
print("  so the HMM's absolute per-sample log-likelihood is naturally lower and NOT directly comparable to GMM.")
print("  We therefore do not pick the model on this number -- it only confirms no model blows up out-of-sample.")
print("- Diagonal vs full HMM land within noise of each other, so the extra parameters of full covariance buy")
print("  nothing out-of-sample; we keep the parsimonious, easily-exported diagonal HMM.")

## 8. Results — the head-to-head scoreboard

Now we put it all in one table: each model scored on geometry (silhouette, Davies–Bouldin,
Calinski–Harabasz), **persistence** (mean run length), **economic separation** (ANOVA F), and
out-of-sample log-likelihood.

When you read it, weight **persistence and economic separation** most heavily — those are the
regime-specific properties. Watch for two classic traps:
- A model that scores great on **silhouette** but has a tiny mean run length is just **flickering**.
- **Agglomerative** can post a huge silhouette by dumping almost everything into one giant cluster plus
  a few outliers — a *degenerate* solution that games the geometry metric while being useless. The
  persistence/economic columns expose this.

In [ ]:
def summarise(name, labels, oos_ll=np.nan):
    # bundle every score for one model into a single row
    m = internal_metrics(X, labels)
    return dict(model=name, **m,
                mean_run_len_days=mean_run_length(labels),   # persistence (days per regime run)
                econ_F=econ_F(labels, fwd_ret),              # economic distinctness (ANOVA F)
                oos_loglik=oos_ll)                           # out-of-sample fit (GMM/HMM only)

comparison = pd.DataFrame([
    summarise("K-Means", km_lab),
    summarise("Agglomerative", ag_lab),
    summarise("GMM", gm_lab, oos_df["GMM"].mean()),
    summarise("HMM", hm_lab, oos_df["HMM (diag)"].mean()),
]).set_index("model")
display(comparison.style.format("{:.3f}").background_gradient(cmap="Greens"))   # greener cell = higher value

print("Higher is better: silhouette, calinski_harabasz, mean_run_len_days, econ_F, oos_loglik")
print("Lower  is better: davies_bouldin")

## 9. Interpreting the regimes — giving the states human names

The HMM hands us numbered states (0, 1, 2, 3) — statistics, not meaning. We make them meaningful by
looking at each state's **economic fingerprint**: its average return and its average volatility.

The labelling rule is simple and explicit:
- highest average return → **Bull / Risk-on**
- lowest average return → **Bear / Risk-off**
- of the two middle states: the *calmer* one → **Accumulation / Recovery**, the *choppier* one →
  **Range / Neutral**

Then we run **Viterbi** (`.predict`) to tag every historical day with its most likely regime, colour
the price chart by regime to eyeball whether it makes sense, and draw the **transition matrix** — a
grid of "probability of going from state X to state Y." A strong **diagonal** there is the visual proof
that regimes *persist* (most of the time tomorrow's state = today's).

In [ ]:
final_hmm = GaussianHMM(**hm_cfg, n_iter=500, random_state=42).fit(X)   # fit the chosen HMM on ALL data
states = final_hmm.predict(X)                                          # Viterbi: most likely regime for each day
model_df = model_df.assign(state=states)

# Economic fingerprint of each state: #days, average return, average volatility, average funding
stats = model_df.groupby("state").agg(
    days=("ret", "size"),
    mean_ret=("ret", "mean"),
    ann_vol=("vol_14", "mean"),
    mean_funding=("funding_ma", "mean"),
)

# Label states by their economic signature: the highest-volatility state is the Capitulation/Crash;
# of the remaining states the lowest average return is the Bear/Downtrend and the highest is Bull;
# the two middle states split by volatility (calmer = Accumulation/Recovery, choppier = Range/Neutral).
crash = stats["ann_vol"].idxmax()
name_map = {crash: "Capitulation / Crash"}
rest_ret = stats.loc[[s for s in stats.index if s != crash]].sort_values("mean_ret")
name_map[rest_ret.index[0]]  = "Bear / Downtrend"
name_map[rest_ret.index[-1]] = "Bull / Risk-on"
mids = stats.loc[list(rest_ret.index[1:-1])].sort_values("ann_vol").index.tolist()
name_map[mids[0]]  = "Accumulation / Recovery"   # calmer of the middle states
name_map[mids[-1]] = "Range / Neutral"           # choppier of the middle states
for s in mids[1:-1]:
    name_map[s] = "Transition"

stats["regime"] = [name_map[s] for s in stats.index]
stats = stats.sort_values("mean_ret")
display(stats)

model_df["regime"] = model_df["state"].map(name_map)
print("Mean regime duration:", round(mean_run_length(states), 1), "days")
print("Regimes found:", ", ".join(name_map[s] for s in stats.index))

In [ ]:
# Plot the BTC price, colouring each day by its decoded regime -> visual sanity check
palette = dict(zip(sorted(model_df["regime"].unique()),
                   ["#f43f5e","#fbbf24","#38bdf8","#34d399","#a78bfa"]))   # one colour per regime
plt.figure(figsize=(15, 5))
for reg, g in model_df.groupby("regime"):
    plt.scatter(g.index, g["close"], s=6, label=reg, color=palette.get(reg))
plt.yscale("log"); plt.title("BTC price coloured by HMM-decoded regime (log scale)")   # log scale so early years show
plt.legend(markerscale=2); plt.tight_layout(); plt.show()

# Transition matrix heatmap: P(go to state j | currently in state i). Strong diagonal = regimes persist.
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(final_hmm.transmat_, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=[f"s{i}" for i in range(N_REGIMES)],
            yticklabels=[f"s{i}" for i in range(N_REGIMES)])
plt.title("HMM transition matrix P(next state | current)")
plt.xlabel("to"); plt.ylabel("from"); plt.tight_layout(); plt.show()
print("Strong diagonal = regimes persist; off-diagonal = transition probabilities.")

## 10. Conclusion & Recommendation — and which model wins where

### The verdict: use the Gaussian **HMM** (diagonal covariance, 5 regimes)
It's the right tool because regimes are fundamentally a **time** phenomenon, and the HMM is the only
candidate that models time. Concretely it wins on the metrics that matter for regimes — **persistence**
(its regimes last ~weeks, not a day) and **economic distinctness** — while also exposing **transition
odds** and being able to **classify new days online**.

### Which model is best *in which situation* (the general lesson)
| Use this | When… | Why |
|---|---|---|
| **K-Means** | quick baseline, round well-separated groups, order doesn't matter | simplest & fastest |
| **Agglomerative** | you want a *hierarchy* / dendrogram, smaller datasets | reveals nested structure |
| **GMM** | clusters overlap & are elliptical, you want soft probabilities, **but time doesn't matter** | flexible probabilistic clustering |
| **HMM** | the data is a **sequence** and states **persist over time** (regimes, speech, gestures) | models transitions + persistence |

The honest summary: **GMM is the close runner-up** — same Gaussian emissions, slightly better raw fit —
but it has *no* temporal memory, so its regimes flicker. K-Means and Agglomerative are baselines that
either flicker or collapse into degenerate clusters. Add the time dimension and the HMM is the clear,
principled choice.

### Getting it into the browser app
The React app can't run Python, so the final cell **exports the trained HMM's parameters** (the scaler,
the start/transition probabilities, and each state's Gaussian) to JSON. In the app we re-implement the
lightweight **forward/Viterbi** inference in TypeScript — at runtime it's just a few matrix multiplies
per day — to classify today's regime live and show its probability and the odds of switching.

In [ ]:
import joblib, os
# Bundle everything the app needs to reproduce the model's inference in TypeScript:
ARTIFACT_DIR = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else "."
export = {
    "features": FEATURES,                          # which features, in order
    "scaler_mean": scaler.mean_.tolist(),          # to standardise live data the SAME way as training
    "scaler_scale": scaler.scale_.tolist(),
    "hmm": {
        "n_states": int(final_hmm.n_components),
        "covariance_type": hm_cfg["covariance_type"],
        "startprob": final_hmm.startprob_.tolist(),    # P(starting in each state)
        "transmat": final_hmm.transmat_.tolist(),      # P(state i -> state j)
        "means": final_hmm.means_.tolist(),            # each state's Gaussian centre
        "covars": [c.tolist() for c in final_hmm.covars_],   # each state's Gaussian spread
    },
    "state_labels": {int(k): v for k, v in name_map.items()},   # state number -> human label
}
with open("regime_hmm_params.json", "w") as fh:
    json.dump(export, fh, indent=2)                 # save as JSON for the browser app
joblib.dump({"hmm": final_hmm, "scaler": scaler, "features": FEATURES}, "regime_hmm_model.joblib")  # save Python objects too
print("Exported regime_hmm_params.json (for the TypeScript engine) and regime_hmm_model.joblib.")
print("Recommended model: Gaussian HMM with config:", hm_cfg)